In [1]:
!pip install zipfile
!pip install pypdf
!pip install dotenv
!pip install pinecone
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-pinecone<0.2" "langchain-text-splitters<0.2"

ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)
ERROR: No matching distribution found for zipfile
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 26.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:0

In [1]:
import os
from google.colab import userdata
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [2]:
import zipfile

ZIP_FILE_DIR = "./sample_data"
ZIP_FILE_NAME = "documents.zip"

with zipfile.ZipFile(ZIP_FILE_DIR + "/" + ZIP_FILE_NAME, 'r') as referencia_zip:
    referencia_zip.extractall(ZIP_FILE_DIR)

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./sample_data/documents/internal_docs_by_area/Customer_support/IronStore_Customer_Support_Escalation_Procedures.pdf")

pages = loader.load()

print(pages[0].page_content)

Customer Support Escalation Procedures
Owning area: Customer_support
Last reviewed: August 2026
Document ID: CS-POL-014
Version: 3.2
Applies to: IronStore customer support operations across Europe
1. Purpose and Overview
This procedure defines how IronStore identifies, manages, and escalates customer contacts that cannot be resolved
through standard frontline support. It is intended to ensure consistent decisions, timely ownership, and appropriate
protection of customer, payment, product, and company information.
Escalation is required when a case involves material customer impact, operational risk, legal or regulatory
considerations, reputational risk, or a resolution outside the agent’s authority. Escalation does not remove ownership
from the original agent unless a receiving team formally accepts the case.
The procedure applies to contacts received through email, telephone, chat, social media, marketplace messaging,
and the IronStore Help Centre.
2. Scope
This procedure covers:
  O

In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [5]:
import re
from pathlib import Path
from langchain_core.documents import Document

def load_clean_document_content(pages):
    page_lines = []
    document_content = []

    for i, page in enumerate(pages):
        page_lines = [re.sub(r'\x7f|Page\s*\d+\s*of\s*\d+\s*—', "", line).strip() for line in page.page_content.split("\n") if line.strip()]
        if len(page_lines):
            document_content.extend(page_lines)
    return document_content

def get_documents_list_content(base_path):
    documents_list = []

    for file in base_path.rglob("*"):
        if file.is_file():
            file_name = file.name;
            if file_name != '.DS_Store':
                loader = PyPDFLoader(file.as_posix())
                pages = loader.load()
                file_path = file.relative_to(base_path).as_posix()
                document_lines = load_clean_document_content(pages)
                documents_list.append(
                    Document(
                        page_content="\n\n".join(document_lines[1:]),
                        metadata={
                            "heading": document_lines[0],
                            "file_name": file_name,
                            "file_path": file_path
                        }
                    )
                )
            else:
                file.unlink()
    return documents_list

In [6]:
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

PINECONE_INDEX_NAME = "enterprise-ai-assistant"
PINECONE_NAMESPACE = "ironhack-documents"

index = pc.Index(PINECONE_INDEX_NAME)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# Create new pinecone index
if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536, # Standard dimensions for OpenAI embeddings
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

documents_list = get_documents_list_content(Path("./sample_data/documents"))

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE
)

for document in documents_list:
  chunks = text_splitter.split_documents([document])

  if chunks:
      vectorstore.add_documents(chunks)

In [7]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only this context:\n\n{context}\n\nQuestion: {input}"
)

combine = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(vectorstore.as_retriever(search_kwargs={"k": 4}), combine)

response = chain.invoke({"input": "What are the usual main working hours?"})
print(response["answer"])

The usual main working hours are from 10:00 to 15:00 local time, Monday to Friday.
